# 🌾 FarmSense — Intelligent Farm-to-Market Decision Engine
## Exploratory Data Analysis (EDA) Notebook

> **Dataset:** 1,635 farm records · 14 features · 6 districts (MP & Maharashtra)  
> **Target:** `market_price_per_kg` (Regression)  
> **Problem:** Predict optimal market price and generate selling recommendations

---

In [ ]:
# ── Cell 1: Setup & Imports ────────────────────────────────
!pip install -q pandas numpy matplotlib seaborn openpyxl plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1b0f',
    'axes.facecolor':   '#0d1b0f',
    'axes.edgecolor':   '#2d4a2d',
    'axes.labelcolor':  '#c8e6c9',
    'xtick.color':      '#a5d6a7',
    'ytick.color':      '#a5d6a7',
    'text.color':       '#e8f5e9',
    'grid.color':       '#1a3a1a',
    'grid.alpha':        0.5,
    'axes.titlesize':   13,
    'axes.titlecolor':  '#81c784',
})

COLORS = ['#4caf50','#f9c74f','#2196f3','#ff7043','#ab47bc',
          '#26c6da','#66bb6a','#ffa726','#ef5350','#78909c']
GOLD  = '#f9c74f'
GREEN = '#4caf50'
RED   = '#ef5350'

print('✅ Setup complete!')

In [ ]:
# ── Cell 2: Load Dataset ────────────────────────────────────
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df_raw = pd.read_excel(filename)
df = df_raw.copy()

print(f'✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   File: {filename}')
df.head()

In [ ]:
# ── Cell 3: Basic Info & Shape ─────────────────────────────
print('=' * 55)
print('  FARMSENSE DATASET — BASIC PROFILE')
print('=' * 55)
print(f'\n📐 Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'💾 Memory  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

print('\n📋 COLUMN TYPES:')
for col in df.columns:
    dtype   = str(df[col].dtype)
    nuniq   = df[col].nunique()
    nmiss   = df[col].isnull().sum()
    miss_s  = f'({nmiss/len(df)*100:.1f}% missing)' if nmiss > 0 else ''
    print(f'   {col:<25} {dtype:<10} {nuniq:>5} unique  {miss_s}')

print('\n📊 NUMERIC SUMMARY:')
display(df.describe().round(2))

In [ ]:
# ── Cell 4: Missing Values ─────────────────────────────────
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})\
            .query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print('MISSING VALUES'); display(miss_df)

fig, ax = plt.subplots(figsize=(9, 4))
colors = [RED if p > 5 else GOLD for p in miss_df['Missing %']]
bars = ax.barh(miss_df.index, miss_df['Missing %'], color=colors, height=0.5, edgecolor='none')
for bar, pct, cnt in zip(bars, miss_df['Missing %'], miss_df['Missing Count']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{cnt} rows ({pct}%)', va='center', fontsize=10, color='#e8f5e9')
ax.set_xlabel('Missing %'); ax.set_title('Missing Values per Column')
ax.set_xlim(0, 12); ax.invert_yaxis()
plt.tight_layout()
plt.savefig('missing_values.png', dpi=150, bbox_inches='tight', facecolor='#0d1b0f')
plt.show()

In [ ]:
# ── Cell 5: Data Quality Issues ────────────────────────────
print('=' * 50)
print('  DATA QUALITY ISSUES')
print('=' * 50)

bad_ph_neg  = df[df['soil_ph'] < 0]
bad_ph_high = df[df['soil_ph'] > 14]
price_out   = df[df['market_price_per_kg'] > 100]

print(f'\n[ERROR] soil_ph < 0  : {len(bad_ph_neg)} rows')
print(bad_ph_neg[['farm_id','crop','district','soil_ph']].to_string(index=False))

print(f'\n[ERROR] soil_ph > 14 : {len(bad_ph_high)} rows')
print(bad_ph_high[['farm_id','crop','district','soil_ph']].to_string(index=False))

print(f'\n[WARN]  market_price > ₹100 : {len(price_out)} rows')
print(price_out[['farm_id','crop','district','market_price_per_kg']].to_string(index=False))

print('\n[WARN]  Crop name values (inconsistency):'); print(df['crop'].value_counts().to_string())
print('\n[WARN]  Season name values (inconsistency):'); print(df['season'].value_counts().to_string())
print(f'\n[INFO]  Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# ── Cell 6: Clean the Data ─────────────────────────────────
df_clean = df.copy()

# Fix crop/season naming
df_clean['crop']   = df_clean['crop'].str.strip().str.title().replace({'Tomatoes': 'Tomato'})
df_clean['season'] = df_clean['season'].str.strip().str.capitalize()

# Fix impossible soil pH
df_clean.loc[df_clean['soil_ph'] < 0,  'soil_ph'] = np.nan
df_clean.loc[df_clean['soil_ph'] > 14, 'soil_ph'] = np.nan

# Drop ID column
df_clean = df_clean.drop(columns=['farm_id'])

# Impute missing values
for col in df_clean.select_dtypes(include=[np.number]).columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print('✅ Cleaning complete!')
print(f'   Rows    : {len(df_clean):,}')
print(f'   Columns : {df_clean.shape[1]}')
print(f'   Missing : {df_clean.isnull().sum().sum()} (should be 0)')
print(f'   Crops   : {sorted(df_clean["crop"].unique())}')
print(f'   Seasons : {sorted(df_clean["season"].unique())}')
df_clean.head()

In [ ]:
# ── Cell 7: Market Price Distribution ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].hist(df_clean['market_price_per_kg'], bins=50, color=GREEN, edgecolor='#0d1b0f', linewidth=0.5)
axes[0].axvline(df_clean['market_price_per_kg'].mean(),   color=GOLD, linestyle='--', lw=2, label=f"Mean: ₹{df_clean['market_price_per_kg'].mean():.2f}")
axes[0].axvline(df_clean['market_price_per_kg'].median(), color=RED,  linestyle='--', lw=2, label=f"Median: ₹{df_clean['market_price_per_kg'].median():.2f}")
axes[0].set_title('Price Distribution (₹/kg)'); axes[0].set_xlabel('Market Price per kg'); axes[0].legend(fontsize=9)

axes[1].boxplot(df_clean['market_price_per_kg'], patch_artist=True,
                boxprops=dict(facecolor=GREEN+'44', color=GREEN),
                medianprops=dict(color=GOLD, linewidth=2),
                whiskerprops=dict(color='#a5d6a7'), capprops=dict(color='#a5d6a7'),
                flierprops=dict(marker='o', color=RED, markersize=3))
axes[1].set_title('Price Boxplot'); axes[1].set_ylabel('₹ per kg')

axes[2].hist(df_clean['market_price_per_kg'], bins=50, color='#26c6da', edgecolor='#0d1b0f', linewidth=0.5, log=True)
axes[2].set_title('Price Distribution (log scale)'); axes[2].set_xlabel('Market Price per kg'); axes[2].set_ylabel('Count (log)')

plt.suptitle('Market Price per kg — Distribution Analysis', fontsize=14, color=GOLD, y=1.02)
plt.tight_layout()
plt.savefig('price_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1b0f')
plt.show()

price = df_clean['market_price_per_kg']
print(f'Mean={price.mean():.2f}  Median={price.median():.2f}  Std={price.std():.2f}  Skew={price.skew():.3f}')

In [ ]:
# ── Cell 8: Price by Crop ──────────────────────────────────
crop_stats = df_clean.groupby('crop')['market_price_per_kg']\
    .agg(['mean','median','std','min','max','count']).round(2).sort_values('mean', ascending=False)
display(crop_stats)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
crops = crop_stats.index.tolist()
means = crop_stats['mean'].values
bar_colors = COLORS[:len(crops)]

bars = axes[0].bar(crops, means, color=bar_colors, edgecolor='none', width=0.6)
axes[0].axhline(df_clean['market_price_per_kg'].mean(), color=GOLD, linestyle='--', lw=1.5, label='Overall avg')
for bar, val in zip(bars, means):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'₹{val:.1f}', ha='center', va='bottom', fontsize=9, color='#e8f5e9')
axes[0].set_title('Average Market Price by Crop'); axes[0].set_ylabel('₹ per kg')
axes[0].legend(fontsize=9); plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

data_per_crop = [df_clean[df_clean['crop']==c]['market_price_per_kg'].values for c in crops]
bp = axes[1].boxplot(data_per_crop, patch_artist=True, labels=crops,
                     medianprops=dict(color=GOLD, linewidth=2),
                     flierprops=dict(marker='o', markersize=2, color=RED))
for patch, color in zip(bp['boxes'], bar_colors):
    patch.set_facecolor(color+'55'); patch.set_edgecolor(color)
axes[1].set_title('Price Distribution by Crop'); axes[1].set_ylabel('₹ per kg')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Market Price Analysis by Crop', fontsize=14, color=GOLD, y=1.02)
plt.tight_layout()
plt.savefig('price_by_crop.png', dpi=150, bbox_inches='tight', facecolor='#0d1b0f')
plt.show()

In [ ]:
# ── Cell 9: Price by District & Season ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dist_means = df_clean.groupby('district')['market_price_per_kg'].mean().sort_values(ascending=True)
colors_d = [GOLD if v == dist_means.max() else GREEN for v in dist_means.values]
axes[0].barh(dist_means.index, dist_means.values, color=colors_d, height=0.5)
axes[0].axvline(df_clean['market_price_per_kg'].mean(), color=RED, linestyle='--', lw=1.5, label='Overall avg')
for i, (d, v) in enumerate(dist_means.items()):
    axes[0].text(v+0.1, i, f'₹{v:.2f}', va='center', fontsize=10, color='#e8f5e9')
axes[0].set_title('Average Price by District'); axes[0].set_xlabel('₹ per kg')
axes[0].legend(fontsize=9); axes[0].set_xlim(0, 36)

seas_means = df_clean.groupby('season')['market_price_per_kg'].mean().sort_values(ascending=False)
axes[1].bar(seas_means.index, seas_means.values, color=[GREEN,GOLD,'#ff7043'][:len(seas_means)], width=0.4)
axes[1].axhline(df_clean['market_price_per_kg'].mean(), color=RED, linestyle='--', lw=1.5, label='Overall avg')
for i, (s, v) in enumerate(seas_means.items()):
    axes[1].text(i, v+0.15, f'₹{v:.2f}', ha='center', fontsize=11, color=GOLD, fontweight='bold')
axes[1].set_title('Average Price by Season'); axes[1].set_ylabel('₹ per kg')
axes[1].legend(fontsize=9); axes[1].set_ylim(0, 35)

plt.suptitle('Price Analysis by District & Season', fontsize=14, color=GOLD, y=1.02)
plt.tight_layout()
plt.savefig('price_by_district_season.png', dpi=150, bbox_inches='tight', facecolor='#0d1b0f')
plt.show()

In [ ]:
# ── Cell 10: Correlation Heatmap ───────────────────────────
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
corr = df_clean[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
cmap = sns.diverging_palette(10, 145, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, center=0,
            annot=True, fmt='.2f', annot_kws={'size': 9},
            linewidths=0.5, linecolor='#0d1b0f',
            square=True, ax=ax, cbar_kws={'shrink': 0.75})
ax.set_title('Correlation Heatmap — All Numeric Features', pad=16)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d1b0f')
plt.show()

print('\n🎯 Correlations with market_price_per_kg:')
target_corr = corr['market_price_per_kg'].drop('market_price_per_kg').sort_values(key=abs, ascending=False)
for feat, val in target_corr.items():
    bar = '█' * int(abs(val) * 30)
    print(f'  {feat:<25} {"+" if val>=0 else "-"}{abs(val):.4f}  {bar}')

In [ ]:
# ── Cell 11: Feature Distributions ────────────────────────
num_feats = [c for c in df_clean.select_dtypes(include=[np.number]).columns if c != 'market_price_per_kg']

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for i, col in enumerate(num_feats):
    ax = axes[i]
    data = df_clean[col].dropna()
    ax.hist(data, bins=40, color=COLORS[i % len(COLORS)], edgecolor='#0d1b0f', linewidth=0.4, alpha=0.85)
    ax.axvline(data.mean(),   color=GOLD, linestyle='--', linewidth=1.5)
    ax.axvline(data.median(), color=RED,  linestyle=':',  linewidth=1.5)
    ax.set_title(f'{col}\n(skew={data.skew():.2f})', fontsize=10)
    ax.set_ylabel('Count')

axes[-1].axis('off')
axes[-1].legend(handles=[
    mpatches.Patch(color=GOLD, label='Mean'),
    mpatches.Patch(color=RED,  label='Median')],
    loc='center', fontsize=12)

plt.suptitle('Distribution of All Numeric Features', fontsize=14, color=GOLD, y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0d1b0f')
plt.show()

In [ ]:
# ── Cell 12: Scatter Plots vs Target ──────────────────────
top_feats = ['yield_tonnes','fertilizer_kg','soil_ph','rainfall_mm','temperature_c','soil_moisture_pct']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(top_feats):
    ax = axes[i]
    corr_val = df_clean[[feat,'market_price_per_kg']].corr().iloc[0,1]
    sc = ax.scatter(df_clean[feat], df_clean['market_price_per_kg'],
                    c=df_clean['market_price_per_kg'], cmap='YlGn',
                    alpha=0.45, s=18, edgecolors='none')
    z = np.polyfit(df_clean[feat].dropna(), df_clean.loc[df_clean[feat].notna(),'market_price_per_kg'], 1)
    x_line = np.linspace(df_clean[feat].min(), df_clean[feat].max(), 200)
    ax.plot(x_line, np.poly1d(z)(x_line), color=GOLD, linewidth=1.5, linestyle='--')
    ax.set_xlabel(feat); ax.set_ylabel('Market Price ₹/kg')
    ax.set_title(f'{feat}\nr = {corr_val:.3f}', fontsize=10)
    plt.colorbar(sc, ax=ax, shrink=0.7)

plt.suptitle('Feature vs Market Price (with trend line)', fontsize=14, color=GOLD, y=1.01)
plt.tight_layout()
plt.savefig('scatter_vs_price.png', dpi=150, bbox_inches='tight', facecolor='#0d1b0f')
plt.show()

In [ ]:
# ── Cell 13: Crop × District Heatmap ──────────────────────
pivot = df_clean.pivot_table(
    values='market_price_per_kg', index='crop', columns='district', aggfunc='mean').round(2)
display(pivot)

fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(pivot.values, cmap='YlGn', aspect='auto')
plt.colorbar(im, ax=ax, label='Avg ₹/kg', shrink=0.75)

ax.set_xticks(range(len(pivot.columns))); ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.columns, fontsize=10); ax.set_yticklabels(pivot.index, fontsize=10)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'₹{val:.1f}', ha='center', va='center',
                    fontsize=8.5, color='black' if val > 28 else 'white')

ax.set_title('Avg Market Price (₹/kg) — Crop × District', pad=14)
ax.set_xlabel('District'); ax.set_ylabel('Crop')
plt.tight_layout()
plt.savefig('crop_district_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ── Cell 14: Outlier Analysis (IQR) ───────────────────────
outlier_summary = []
for col in df_clean.select_dtypes(include=[np.number]).columns:
    Q1, Q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    outlier_summary.append({
        'Column': col, 'Q1': round(Q1,2), 'Q3': round(Q3,2),
        'IQR': round(IQR,2), 'Lower': round(lower,2), 'Upper': round(upper,2),
        'Outliers': n_out, 'Outlier %': round(n_out/len(df_clean)*100, 2)
    })

out_df = pd.DataFrame(outlier_summary).sort_values('Outlier %', ascending=False)
display(out_df)

sig = out_df[out_df['Outlier %'] > 0]
fig, ax = plt.subplots(figsize=(10, 4))
colors = [RED if p > 5 else GOLD for p in sig['Outlier %']]
ax.bar(sig['Column'], sig['Outlier %'], color=colors, edgecolor='none', width=0.5)
ax.set_title('Outlier Percentage per Feature (IQR method)')
ax.set_ylabel('Outlier %')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
for i, (_, row) in enumerate(sig.iterrows()):
    ax.text(i, row['Outlier %']+0.1, f"{row['Outlier %']}%",
            ha='center', va='bottom', fontsize=9, color='#e8f5e9')
plt.tight_layout()
plt.savefig('outliers.png', dpi=150, bbox_inches='tight', facecolor='#0d1b0f')
plt.show()

In [ ]:
# ── Cell 15: Feature Engineering ──────────────────────────
df_fe = df_clean.copy()

df_fe['yield_per_acre']    = df_fe['yield_tonnes'] / df_fe['farm_area_acres']
df_fe['temp_stress']       = (df_fe['temperature_c'] - 25).abs() / 5
df_fe['soil_health_score'] = 1 / (1 + (df_fe['soil_ph'] - 6.5).abs())
df_fe['water_per_tonne']   = df_fe['water_usage_litres'] / df_fe['yield_tonnes']
df_fe['fert_per_acre']     = df_fe['fertilizer_kg'] / df_fe['farm_area_acres']
df_fe['season_code']       = df_fe['season'].map({'Kharif':1,'Rabi':2,'Summer':3})

print('✅ Feature Engineering complete!')
print(f'   Original: {df_clean.shape[1]} columns → After: {df_fe.shape[1]} columns')

new_feats = ['yield_per_acre','temp_stress','soil_health_score','water_per_tonne','fert_per_acre','season_code']
print('\n   Correlation of new features with market price:')
corr_new = df_fe[new_feats + ['market_price_per_kg']].corr()['market_price_per_kg'].drop('market_price_per_kg')
display(corr_new.sort_values(key=abs, ascending=False).to_frame())

In [ ]:
# ── Cell 16: EDA Summary ───────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════╗
║         FARMSENSE — EDA COMPLETE SUMMARY                ║
╚══════════════════════════════════════════════════════════╝

📦 DATASET
   Rows    : 1,635
   Columns : 14 (13 after dropping farm_id)
   Target  : market_price_per_kg (REGRESSION)

❓ MISSING VALUES (fixed by median/mode imputation)
   crop         : 34 rows (2.08%)
   soil_ph      : 33 rows (2.02%)
   rainfall_mm  : 40 rows (2.45%)
   yield_tonnes : 33 rows (2.02%)

⚠️  DATA QUALITY ISSUES FOUND & FIXED
   soil_ph < 0       : 3 rows   → set to NaN then imputed
   soil_ph > 14      : 1 row    → set to NaN then imputed
   Tomatoes → Tomato : naming standardised
   RABI/kharif       : season names normalised to title case

💰 TARGET — market_price_per_kg
   Mean   : ₹28.69   |   Std    : ₹23.75  (HIGH — outliers)
   Median : ₹24.33   |   Skew   : right-skewed
   Min    : ₹6.61    |   Max    : ₹419.04

🏆 KEY FINDINGS
   Best district  → Dewas   ₹30.85/kg
   Best crop      → Potato  ₹33.80/kg
   Best season    → Kharif  ₹29.04/kg
   Worst district → Bhopal  ₹27.18/kg
   Worst crop     → Cotton  ₹26.73/kg

🤖 RECOMMENDED MODEL
   Type    : Regression
   Best    : XGBoost > GradientBoosting > RandomForest
   Metric  : RMSE + R²
   Key tip : Crop type & District are top price drivers
             (use OneHotEncoder — they're categorical!)
""")

import os
pngs = sorted([f for f in os.listdir('.') if f.endswith('.png')])
print(f'📁 {len(pngs)} chart(s) saved: {pngs}')